# 04 — Official results and interpretation

This is the reproducible research-run notebook. It invokes the one official pipeline, then reads the saved tables rather than recalculating metrics in notebook cells. For final results, record the Git commit, package versions, and command settings with the paper.

## Before running

1. Complete `02_demand.ipynb` and review its data-quality output.
2. Review the study assumptions in `src/config.py`.
3. Confirm that `data/reference/lptrp.json` is the allocation you intend to evaluate.
4. Keep `RUN_PRODUCTION_PIPELINE` false until you are ready; a production run uses 200 bootstrap samples and may take time.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)

from run_pipeline import run
from src.paths import ProjectPaths

paths = ProjectPaths.discover()
try:
    revision = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
except (OSError, subprocess.CalledProcessError):
    revision = 'unavailable'

pd.Series({'python': sys.version.split()[0], 'platform': platform.platform(), 'git_revision': revision})

In [ ]:
RUN_PRODUCTION_PIPELINE = False
BOOTSTRAP_SAMPLES = 200
MULTISTART_RUNS = 10
INCLUDE_MAPS = True

if RUN_PRODUCTION_PIPELINE:
    results = run(
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        multistart_runs=MULTISTART_RUNS,
        skip_maps=not INCLUDE_MAPS,
    )
else:
    print('No pipeline run started. Set RUN_PRODUCTION_PIPELINE = True when ready.')

## Read saved results

The remaining cells work after a completed pipeline run. They make the interpretation trail explicit: allocation → welfare → robustness → bottlenecks.

In [ ]:
required_tables = [
    'table1_welfare_comparison.csv', 'table2_bottleneck_analysis.csv',
    'table3_driver_allocation.csv', 'table5_sensitivity_analysis.csv',
]
missing = [name for name in required_tables if not (paths.tables / name).exists()]
assert not missing, f'Missing result tables: {missing}. Run the pipeline first.'

welfare = pd.read_csv(paths.tables / 'table1_welfare_comparison.csv')
allocation = pd.read_csv(paths.tables / 'table3_driver_allocation.csv')
display(welfare)
allocation

In [ ]:
sensitivity = pd.read_csv(paths.tables / 'table5_sensitivity_analysis.csv')
bottlenecks = pd.read_csv(paths.tables / 'table2_bottleneck_analysis.csv')
display(sensitivity)
bottlenecks.sort_values('marginal_welfare_gain', ascending=False)

## Interpretation checklist

- **Price of Anarchy** compares decentralized driver choices against the study's social optimum. Values above 1 indicate a gap.
- **LPTRP improvement ratio** reports how much of that gap the proposal closes: 1 closes all of it; 0 does not improve it; a negative value worsens it.
- **Sensitivity** asks whether the conclusion holds when the welfare weight, demand, or fuel-cost assumptions change.
- **Bottlenecks** rank the modelled welfare gain from one added jeepney. They are decision-support signals, not a substitute for policy review.

In [ ]:
report_path = paths.reports / 'summary_report.txt'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))
else:
    print('Summary report will appear after the pipeline completes.')